[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/33_beam_search.ipynb)

# 🟠 Medium: Beam Search Decoding

Implement **beam search** — the classic decoding algorithm for sequence generation.

### Signature
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: takes token list, returns (V,) log-probabilities
    # Returns: best sequence (list of ints)
```

### Algorithm
1. Start with `[(0.0, [start_token])]`
2. Each step: expand each beam with top-k next tokens
3. Keep top `beam_width` beams by total log-probability
4. Stop when best beam ends with `eos_token` or `max_len` reached

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [3]:
import torch

In [12]:
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    # beams: list of (score, tokens)
    beams = [(0.0, [start_token])]

    for _ in range(max_len):
        candidates = []
        for score, tokens in beams:
            if tokens[-1] == eos_token:
                candidates.append((score, tokens))
                continue

            # 扩展当前 beam
            log_probs = log_prob_fn(tokens) # (V,)
            top_lp, top_idx = torch.topk(log_probs, beam_width)

            for lp, idx in zip(top_lp, top_idx):
                candidates.append((score + lp.item(), tokens + [idx.item()]))

        # 筛选全局前 k 个
        beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]

        # Best beam 结束则退出
        if beams[0][1][-1] == eos_token:
            break

    return beams[0][1]

In [10]:
# 🧪 Debug
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('Sequence:', seq)

Sequence: [0, 1, 2, 3, 4]


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('beam_search')


🧪 Testing: Beam Search Decoding (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Returns list starting with start_token (1.7ms)
  ✅ [2/4] Greedy path (beam=1) (0.7ms)
  ✅ [3/4] Beam finds better path than greedy (0.6ms)
  ✅ [4/4] Stops at eos (0.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (3.5ms total)
  Progress saved. Run status() to see your dashboard.

